# P16: a cheaper guide for the kiosk

The kiosks run on modest CPUs (the brief). P09 timed the final model at 70 ms per photo on one thread of the computer it ran
on. Two ways to make it cheaper, without going back to the weaker models:

1. **int8 quantization of the final model.** Its weights and activations are stored as 8-bit integers instead of 32-bit
   floats: four times smaller, and integer arithmetic is faster on most CPUs. Post-training static quantization (PyTorch FX):
   the network is first run on 200 training paintings to measure the range of every activation, then converted.
2. **A network designed for phones: MobileNetV3** (Large and Small, pretrained on ImageNet, from torchvision). It uses
   depthwise convolutions, far cheaper than the ResNet's. As for the frozen ResNet-18 of P06: the whole network frozen, a new
   final layer trained on our paintings.

**The rules, fixed before any result** (a scratch run, unsaved, checked the code and the run times):

- **int8** is accepted if its validation accuracy is within 1 point of the final model's (83.3%); if so, it is scored once
  on museum B.
- **MobileNetV3**: 5 seeds, 30 epochs, Adam at 1e-3 (the frozen ResNet-18 settings), saved with `run_experiment`. A MobileNet
  is scored once on museum B if its 5-seed validation mean is within 2 points of the final model (at least 81.3%).
- **Cost:** every network is timed here, with the protocol of P09 (one image at a time, interleaved rounds, median), on one
  thread and on all the threads of this computer, and its size is the size of its saved weights. The milliseconds belong to
  this computer (P09's was another one): compare the ratios between networks.

In [1]:
import copy
import io
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.ao.quantization import get_default_qconfig_mapping
from torch.ao.quantization.quantize_fx import convert_fx, prepare_fx

from deep_learning_project_waerniers_lamy_mahieddine.vision.utils import (
    RESULTS_DIR, build_cnn, build_dense, build_resnet_head, final_model, load_resnet_data, predict_logits, pretrained_resnet,
    run_experiment, score_on_test, split_images, summary, time_per_image, to_tensor)

ENGINE = "fbgemm" if "fbgemm" in torch.backends.quantized.supported_engines else "qnnpack"
torch.backends.quantized.engine = ENGINE  # fbgemm on x86 processors, qnnpack on ARM ones (this computer: see below)
print("quantized engine:", ENGINE, "| threads available:", torch.get_num_threads())


def size_mb(model):
    """Size of the saved weights, in MB."""
    buffer = io.BytesIO()
    torch.save(model.state_dict(), buffer)
    return buffer.tell() / 1e6


fp32 = final_model()
val_images, val_labels = split_images("val")
fp32_val_logits = predict_logits(fp32, val_images)
fp32_val = (fp32_val_logits.argmax(1) == val_labels).float().mean().item()
print(f"final model (float32): validation {fp32_val:.1%}, {size_mb(fp32):.1f} MB")

quantized engine: qnnpack | threads available: 4


final model (float32): validation 83.3%, 44.8 MB


## 1. int8 quantization of the final model

In [2]:
import warnings

train_images, _ = split_images("train")
calibration = [train_images[i] for i in torch.randperm(len(train_images), generator=torch.Generator().manual_seed(0))[:200].tolist()]
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # torch.ao.quantization announces its move to the torchao package
    prepared = prepare_fx(copy.deepcopy(fp32), get_default_qconfig_mapping(ENGINE), example_inputs=(to_tensor(calibration[:1]),))
    with torch.no_grad():
        for i in range(0, len(calibration), 50):  # measure the range of every activation on 200 training paintings
            prepared(to_tensor(calibration[i:i + 50]))
    int8 = convert_fx(prepared)

int8_val_logits = predict_logits(int8, val_images)
int8_val = (int8_val_logits.argmax(1) == val_labels).float().mean().item()
agreement = (int8_val_logits.argmax(1) == fp32_val_logits.argmax(1)).float().mean().item()
int8_accepted = int8_val >= fp32_val - 0.01
print(f"int8: validation {int8_val:.1%} (float32 {fp32_val:.1%}); same prediction as float32 on {agreement:.1%} of the paintings; "
      f"{size_mb(int8):.1f} MB against {size_mb(fp32):.1f} MB")
print("rule (within 1 point of float32):", "accepted" if int8_accepted else "rejected")

int8: validation 83.3% (float32 83.3%); same prediction as float32 on 98.1% of the paintings; 11.2 MB against 44.8 MB
rule (within 1 point of float32): accepted


**What we see.**

- **int8 costs nothing measurable in accuracy:** 83.3% on validation, as in float32, and the same predicted genre for 98.1% of the paintings (9 of 480 change, with no net effect on the accuracy). The weights go from 44.8 MB to 11.2 MB. The rule is met: it is scored on museum B in section 4.
- This computer has an ARM processor, so the quantized kernels are those of `qnnpack`; an x86 kiosk would use `fbgemm`. The notebook picks the engine itself.